In [1]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
pd.options.mode.chained_assignment = None


# sctrial: Scalability Analysis


## 1. Benchmarking


In [2]:
import scanpy as sc
import pandas as pd
import numpy as np
import sctrial as st
import time

adata = sc.datasets.pbmc68k_reduced()
adata.obs['donor'] = np.random.choice([f"D{i}" for i in range(20)], size=adata.n_obs)
adata.obs['visit'] = np.random.choice(['V1', 'V2'], size=adata.n_obs)
adata.obs['arm'] = np.where(adata.obs['donor'].isin([f"D{i}" for i in range(10)]), 'T', 'C')

c = adata.obs.groupby(['donor', 'visit'], observed=True).size().unstack(fill_value=0)
keep = c[(c.get('V1',0)>0) & (c.get('V2',0)>0)].index
adata = adata[adata.obs['donor'].isin(keep)].copy()

start = time.time()
res = st.did_table(adata, features=adata.var_names[:1000], 
                   design=st.TrialDesign(participant_col='donor', visit_col='visit', arm_col='arm', 
                                         arm_treated='T', arm_control='C'), 
                   visits=('V1', 'V2'), aggregate='participant_visit')
print(f"Genome-wide DiD (1000 genes) took {time.time() - start:.2f} seconds.")


Genome-wide DiD (1000 genes) took 2.10 seconds.
